# Flash Attention for Lorentz Attention in HyperCore

A proof-of-concept Triton kernel for the `full` attention path of HyperCore's
`LorentzMultiheadAttention`, which normally materializes the full `[B, H, N, N]`
score matrix (quadratic memory).

The idea behind FlashAttention transfers to Lorentz attention almost unchanged, because:

1. the Lorentzian inner product `<q,k>_L = -q0*k0 + <q_space, k_space>` is just a dot
   product after flipping the sign of the time coordinate;
2. the `+2c` and `+bias` terms in the score are constant shifts, which softmax ignores;
3. the value aggregation `softmax(S) @ V` is linear, so the online-softmax recurrence is
   exact, and the only nonlinear step (reprojecting the result back onto the hyperboloid)
   happens once at the very end.

So we never need to build the `N x N` matrix: memory drops from O(N^2) to O(N).

This notebook defines the kernel, patches it into the real HyperCore layer, checks it is
numerically identical to the original, and benchmarks it (speed + peak memory).

**Run on a GPU runtime (Tesla T4).**

## 1. Setup

Clone HyperCore and install the light dependencies. `torch` / `triton` are already on Kaggle.

In [ ]:
import os
import sys
import subprocess

if not os.path.isdir("HyperCore"):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/Graph-and-Geometric-Learning/HyperCore.git"],
        check=True,
    )

for pkg in ("geoopt", "loguru", "einops"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

sys.path.insert(0, "HyperCore")

## 2. Lorentz helpers

Two small operations we reuse everywhere. A point on the hyperboloid is stored as
`x = (x0, x_space)` with the time coordinate `x0` at index 0.

In [ ]:
import math
import torch


def lorentz_self_inner(v):
    
    space = v[..., 1:]
    time = v[..., :1]
    return (space * space).sum(-1, keepdim=True) - time * time


def reproject(ave, c, eps=1e-15):
    
    denom = lorentz_self_inner(ave).abs().clamp_min(eps).sqrt()
    return math.sqrt(c) * ave / denom

## 3. Reference implementation (pure PyTorch)

A block-wise version with the online-softmax recurrence. It never builds the `N x N`
matrix, so it is already O(N) in memory, and it is fully differentiable. We treat this as
the *ground truth* the Triton kernel must match, and as the fallback for the cases the
kernel does not handle (masks, training).

In [ ]:
def flash_torch(q, k, v, c, scale, block_n=128, mask=None):
    
    B, H, N, Dt = q.shape
    inv_scale = 2.0 / scale 
    
    q_neg = q.clone()
    q_neg[..., 0] = -q_neg[..., 0]

    o = q.new_zeros(B, H, N, Dt)              
    m = q.new_full((B, H, N, 1), float("-inf")) 
    l = q.new_zeros(B, H, N, 1)                

    mask_is_bool = mask is not None and mask.dtype == torch.bool

    for j0 in range(0, N, block_n):
        j1 = min(j0 + block_n, N)
        k_block = k[:, :, j0:j1]
        v_block = v[:, :, j0:j1]

        scores = torch.matmul(q_neg, k_block.transpose(-1, -2)) * inv_scale

        if mask is not None:
            m_block = mask[:, :, :, j0:j1]
            if mask_is_bool:
                scores = scores.masked_fill(m_block, float("-inf"))
            else:
                scores = scores + m_block

        m_new = torch.maximum(m, scores.max(dim=-1, keepdim=True).values)
        p = torch.exp(scores - m_new)
        alpha = torch.exp(m - m_new)

        l = alpha * l + p.sum(dim=-1, keepdim=True)
        o = alpha * o + torch.matmul(p, v_block)
        m = m_new

    ave = o / l.clamp_min(1e-20)
    return reproject(ave, c)

## 4. Triton kernel (fp32)

The fast path. Two things make this work on a T4:

- **No padding.** The feature size is `D + 1 = 65`, and Triton tile sizes must be powers of
  two, which would round 65 up to 128 and double the work. Instead we split the time
  coordinate out: do the dot product over the `D = 64` *space* dimensions (already a power
  of two) and add the rank-1 term `-q0*k0` separately. Same for the value aggregation.
- **Shared-memory fallback.** The T4 has only 64 KB of shared memory per SM, so large tiles
  can fail to launch. We try a few configurations from fast to safe and fall back on an
  out-of-resources error, caching whatever fits.

In [ ]:
HAS_TRITON = False
try:
    import triton
    import triton.language as tl
    HAS_TRITON = torch.cuda.is_available()
except ImportError:
    pass


if HAS_TRITON:

    @triton.jit
    def _flash_lorentz_fwd(
        Q, K, V, O, sm_scale,
        stride_qh, stride_qn, stride_qd,
        stride_kh, stride_kn, stride_kd,
        stride_vh, stride_vn, stride_vd,
        stride_oh, stride_on, stride_od,
        N,
        D: tl.constexpr,
        BLOCK_M: tl.constexpr,
        BLOCK_N: tl.constexpr,
        BLOCK_D: tl.constexpr,
    ):
        start_m = tl.program_id(0)
        off_h = tl.program_id(1)

        offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_d = tl.arange(0, BLOCK_D)  
        mask_d = offs_d < D
        mask_m = offs_m < N
        
        q_space = tl.load(
            Q + off_h * stride_qh + offs_m[:, None] * stride_qn + (1 + offs_d)[None, :] * stride_qd,
            mask=mask_m[:, None] & mask_d[None, :], other=0.0,
        )
        q_time = tl.load(Q + off_h * stride_qh + offs_m * stride_qn, mask=mask_m, other=0.0)

        m_i = tl.full([BLOCK_M], float("-inf"), tl.float32)
        l_i = tl.zeros([BLOCK_M], tl.float32)
        acc_space = tl.zeros([BLOCK_M, BLOCK_D], tl.float32)
        acc_time = tl.zeros([BLOCK_M], tl.float32)

        for n0 in range(0, N, BLOCK_N):
            offs_n = n0 + tl.arange(0, BLOCK_N)
            mask_n = offs_n < N

            k_space = tl.load(
                K + off_h * stride_kh + offs_n[:, None] * stride_kn + (1 + offs_d)[None, :] * stride_kd,
                mask=mask_n[:, None] & mask_d[None, :], other=0.0,
            )
            v_space = tl.load(
                V + off_h * stride_vh + offs_n[:, None] * stride_vn + (1 + offs_d)[None, :] * stride_vd,
                mask=mask_n[:, None] & mask_d[None, :], other=0.0,
            )
            k_time = tl.load(K + off_h * stride_kh + offs_n * stride_kn, mask=mask_n, other=0.0)
            v_time = tl.load(V + off_h * stride_vh + offs_n * stride_vn, mask=mask_n, other=0.0)

            scores = tl.dot(q_space, tl.trans(k_space)) - q_time[:, None] * k_time[None, :]
            scores = scores * sm_scale
            scores = tl.where(mask_n[None, :], scores, float("-inf"))

            m_new = tl.maximum(m_i, tl.max(scores, axis=1))
            p = tl.exp(scores - m_new[:, None])
            alpha = tl.exp(m_i - m_new)

            l_i = alpha * l_i + tl.sum(p, axis=1)
            acc_space = acc_space * alpha[:, None] + tl.dot(p.to(v_space.dtype), v_space)
            acc_time = acc_time * alpha + tl.sum(p * v_time[None, :], axis=1)
            m_i = m_new

        acc_space = acc_space / l_i[:, None]
        acc_time = acc_time / l_i

        tl.store(
            O + off_h * stride_oh + offs_m[:, None] * stride_on + (1 + offs_d)[None, :] * stride_od,
            acc_space, mask=mask_m[:, None] & mask_d[None, :],
        )
        tl.store(O + off_h * stride_oh + offs_m * stride_on, acc_time, mask=mask_m)

### Launcher

Reshapes to `[B*H, N, D+1]`, picks a working block configuration, and applies the
reprojection (cheap, done in PyTorch).

In [ ]:
if HAS_TRITON:

    _CONFIGS = [
        dict(block_m=128, block_n=64, num_stages=2, num_warps=4),
        dict(block_m=64,  block_n=64, num_stages=2, num_warps=4),
        dict(block_m=64,  block_n=32, num_stages=2, num_warps=4),
        dict(block_m=64,  block_n=32, num_stages=1, num_warps=4),
        dict(block_m=32,  block_n=32, num_stages=1, num_warps=2),
    ]
    _best_config = {} 

    def flash_triton(q, k, v, c, scale):
        B, H, N, Dt = q.shape
        D = Dt - 1

        q2 = q.reshape(B * H, N, Dt).contiguous()
        k2 = k.reshape(B * H, N, Dt).contiguous()
        v2 = v.reshape(B * H, N, Dt).contiguous()
        out = torch.empty_like(q2)

        block_d = triton.next_power_of_2(D)
        sm_scale = 2.0 / scale

        configs = [_best_config[Dt]] if Dt in _best_config else _CONFIGS
        last_error = None
        for cfg in configs:
            try:
                grid = (triton.cdiv(N, cfg["block_m"]), B * H)
                _flash_lorentz_fwd[grid](
                    q2, k2, v2, out, sm_scale,
                    q2.stride(0), q2.stride(1), q2.stride(2),
                    k2.stride(0), k2.stride(1), k2.stride(2),
                    v2.stride(0), v2.stride(1), v2.stride(2),
                    out.stride(0), out.stride(1), out.stride(2),
                    N, D,
                    BLOCK_M=cfg["block_m"], BLOCK_N=cfg["block_n"], BLOCK_D=block_d,
                    num_warps=cfg["num_warps"], num_stages=cfg["num_stages"],
                )
                _best_config[Dt] = cfg
                return reproject(out.reshape(B, H, N, Dt), c)
            except Exception as e:
                if "out of resource" in str(e).lower() or "shared memory" in str(e).lower():
                    last_error = e
                    continue
                raise
        raise RuntimeError(f"No Triton config fit in shared memory; last error: {last_error}")

## 5. Import the real HyperCore layer

Importing `hypercore` runs its package `__init__`, which pulls in graph/PEFT modules that
need `torch_scatter`, `transformers`, etc. We do not use any of that here (only attention,
manifolds, and conv), so the helper below stubs out whatever is missing and retries the
import, instead of forcing you to install heavy GPU-compiled packages.

In [ ]:
import importlib
import importlib.abc
import importlib.machinery
import types


class _DummyBase:
    
    def __init__(self, *args, **kwargs): pass
    def __call__(self, *args, **kwargs): return None
    def __getattr__(self, name): return _DummyBase()


class _StubModule(types.ModuleType):
    
    def __init__(self, name):
        
        super().__init__(name)
        self.__path__ = []        
        self._cache = {}

    def __getattr__(self, name):
        
        if name.startswith("__") and name.endswith("__"):
            raise AttributeError(name)
        
        if name not in self._cache:
            self._cache[name] = type("_Dummy_" + name, (_DummyBase,), {})
        return self._cache[name]


_BLOCKED = set()


class _StubLoader(importlib.abc.Loader):
    def create_module(self, spec):
        return _StubModule(spec.name)

    def exec_module(self, module):
        pass


class _StubFinder(importlib.abc.MetaPathFinder):
    def find_spec(self, name, path, target=None):
        if name.split(".")[0] in _BLOCKED:
            return importlib.machinery.ModuleSpec(name, _StubLoader())
        return None


sys.meta_path.insert(0, _StubFinder())


def import_hypercore_attention():
    
    for _ in range(50):
        try:
            import hypercore.manifolds as manifolds
            from hypercore.nn.attention.lorentz_former_conv import LorentzMultiheadAttention
            return manifolds.Lorentz, LorentzMultiheadAttention
        except ModuleNotFoundError as e:
            missing = (e.name or "").split(".")[0]
            if not missing or missing in _BLOCKED:
                raise
            _BLOCKED.add(missing)
            # Drop half-initialized modules so the retry starts clean.
            for mod in [m for m in list(sys.modules) if m.split(".")[0] == missing]:
                del sys.modules[mod]
    raise RuntimeError("stubbing too many modules")


Lorentz, LorentzMultiheadAttention = import_hypercore_attention()
if _BLOCKED:
    print("Stubbed out (means that unused here):", sorted(_BLOCKED))

## 6. Patch `full_attention`

A drop-in replacement for the body of `full_attention`. It reuses the original projection,
normalization, and head-merge code verbatim and only swaps the O(N^2) core for the flash
kernel. `patch(True)` installs it, `patch(False)` restores the original, so we can A/B the
*same layer*.

The Triton path is used for plain inference; anything with a mask or that needs gradients
goes through the differentiable PyTorch reference.

In [ ]:
def _flash_full_attention(self, qs, ks, vs, output_attentions=False, mask=None):

    if output_attentions:
        return self._orig_full_attention(qs, ks, vs, True, mask)

    from hypercore.nn.conv import LorentzNormalization

    qs = self.project(qs)
    ks = self.project(ks)
    vs = self.project(vs)
    if self.normalize:
        qs = LorentzNormalization(self.manifold)(qs)
        ks = LorentzNormalization(self.manifold)(ks)

    q = qs.transpose(1, 2)  
    k = ks.transpose(1, 2)
    v = vs.transpose(1, 2)

    c = float(self.manifold.c)
    use_triton = HAS_TRITON and q.is_cuda and mask is None and not torch.is_grad_enabled()
    if use_triton:
        att_output = flash_triton(q, k, v, c, float(self.scale))
    else:
        
        att_output = flash_torch(q, k, v, c, self.scale, mask=mask)

    att_output = att_output.transpose(1, 2) 

    if self.trans_heads_concat:
        space = self.final_linear(
            att_output.reshape(att_output.size(0), att_output.size(1),
                               self.num_heads * self.out_channels)
        )
        time = ((space ** 2).sum(dim=-1, keepdims=True) + self.manifold.c).sqrt()
        return torch.cat([time, space], dim=-1)
    return self.manifold.lorentzian_centroid(att_output)


def patch(enable):
    
    already = getattr(LorentzMultiheadAttention, "_patched", False)
    if enable and not already:
        LorentzMultiheadAttention._orig_full_attention = LorentzMultiheadAttention.full_attention
        LorentzMultiheadAttention.full_attention = _flash_full_attention
        LorentzMultiheadAttention._patched = True
    elif not enable and already:
        LorentzMultiheadAttention.full_attention = LorentzMultiheadAttention._orig_full_attention
        LorentzMultiheadAttention._patched = False

## 7. Device and a small layer factory

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU",
      "| triton:", HAS_TRITON)

manifold = Lorentz(c=1.0)


def make_layer(num_heads=8, in_channels=16, out_channels=65, concat=True):
    
    torch.manual_seed(0)
    return LorentzMultiheadAttention(
        manifold, in_channels, out_channels, num_heads,
        attention_type="full", trans_heads_concat=concat,
    ).to(DEVICE)

## 8. Correctness

Run the *same* layer twice on the same input -- once with the original `full_attention`,
once with the flash kernel patched in -- and compare. The only difference between the two
runs is the attention core, so any gap is purely the kernel's error.

In [ ]:
for concat in (False, True):
    layer = make_layer(concat=concat)
    if DEVICE == "cpu":
        layer = layer.double()

    batch, seq_len = 2, 256
    feat = layer.num_heads * layer.in_channels
    dtype = torch.float64 if DEVICE == "cpu" else torch.float32
    x = torch.randn(batch, seq_len, feat, dtype=dtype, device=DEVICE)

    patch(False)
    with torch.no_grad():
        out_original = layer(x, x)

    patch(True)
    with torch.no_grad():
        out_flash = layer(x, x)
    patch(False)

    err = (out_original - out_flash).abs().max().item()
    print(f"  concat={concat!s:<5}  max abs error = {err:.2e}   output shape {tuple(out_original.shape)}")

## 9. Benchmark

Speed (`torch.cuda.Event`, median of 100 runs with min--max) and peak memory
(`max_memory_allocated`), comparing the real `full_attention` against the flash kernel on
the same layer. We sweep a few head/dim configurations because the speedup depends on the
head dimension `D`.

Note this is forward-only, batch size 1, a single layer. Larger batch, full models, and the
backward pass are the natural next measurements.

In [ ]:
import statistics

def time_ms(fn, iters=100, warmup=20):
    
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    samples = []
    for _ in range(iters):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        end.record()
        torch.cuda.synchronize()
        samples.append(start.elapsed_time(end))

    samples.sort()
    return statistics.median(samples), samples[0], samples[-1]


def peak_memory_mb(fn):
    
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    try:
        fn()
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / 1e6
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            return -1.0
        raise

In [ ]:

for num_heads, D in [(8, 64), (12, 64), (4, 32)]:
    layer = make_layer(num_heads=num_heads, out_channels=D + 1)
    feat = layer.num_heads * layer.in_channels

    print(f"\n--- H={num_heads}, D={D} ---")
    header = (f"{'N':>6}  {'orig ms':>22}  {'flash ms':>22}  {'speedup':>8}  "
              f"{'orig MB':>9}  {'flash MB':>9}  {'memx':>6}")
    print(header)
    print("-" * len(header))

    for N in [1024, 2048, 4096, 8192, 16384]:
        x = torch.randn(1, N, feat, device=DEVICE)

        def run_original():
            patch(False)
            with torch.no_grad():
                return layer(x, x)

        def run_flash():
            patch(True)
            with torch.no_grad():
                return layer(x, x)

        mem_orig = peak_memory_mb(run_original)
        mem_flash = peak_memory_mb(run_flash)

        if mem_orig < 0: 
            med, lo, hi = time_ms(run_flash)
            flash_str = f"{med:.2f} ({lo:.2f}-{hi:.2f})"
            print(f"{N:>6}  {'OOM':>22}  {flash_str:>22}  {'--':>8}  "
                  f"{'OOM':>9}  {mem_flash:>9.1f}  {'inf':>6}")
        else:
            mo, lo_o, hi_o = time_ms(run_original)
            mf, lo_f, hi_f = time_ms(run_flash)
            orig_str = f"{mo:.2f} ({lo_o:.2f}-{hi_o:.2f})"
            flash_str = f"{mf:.2f} ({lo_f:.2f}-{hi_f:.2f})"
            print(f"{N:>6}  {orig_str:>22}  {flash_str:>22}  {mo / mf:>7.2f}x  "
                  f"{mem_orig:>9.1f}  {mem_flash:>9.1f}  {mem_orig / mem_flash:>5.1f}x")

        del x
        torch.cuda.empty_cache()

    patch(False)

## Notes

- **Memory** is the unconditional win: flat O(N) instead of quadratic, so the baseline OOMs
  at long sequences while flash keeps running.
- **Speed** depends on the head dimension. For `D=64` flash is faster at every length; for a
  small head (`D=32`) the space dot product is too small to amortize the kernel overhead and
  the baseline's small matmul is hard to beat.
- This is fp32. Using the T4 tensor cores would need fp16/bf16, but Triton's `tl.dot` does
  not emit MMA instructions on sm_75 (Turing) -- which is the motivation for a hand-written
  CUDA (`wmma`) kernel as the next step.